
# List available models and configs for a solver

Inspect what ``incompressibleFluid`` knows about *without* running it:
which plugin models are registered, what config fields each one expects,
and which ``fvSchemes`` / ``fvSolution`` entries the active operations
require.

This is the entry point for UI generation, AI-assisted case templating,
and writing your own pre-flight checks. Everything below stops short of
the ``BUILD`` stage — no mesh is loaded, no fields are allocated.

If you haven't run a NeoFOAM case yet, do
:doc:`/auto_tutorials/example_01_run_incompressible_fluid` first.


## Imports



In [ ]:
import json
from pathlib import Path

from neofoam.core.plugin_system import PluginSystem
from neofoam.solver.incompressibleFluid.create_fields import create_init

# Importing the solver's plugin interface ensures the registry is
# populated before we ask ``PluginSystem`` what's available. Models
# that live in third-party packages only show up after their package
# has been imported.
from neofoam.solver.incompressibleFluid.models import (  # noqa: F401
    incompressibleFluidModel,
)
from neofoam.tutorial import clone_case

## Enumerate registered plugins (no case needed)
``PluginSystem.get_registered`` returns the registry for a named
plugin interface. ``plugin_registry`` is the list of every class
that called ``Model("...").register_with(incompressibleFluidModel)``.
This works without any case directory and answers the question
*"which optional models could a case opt into?"*



In [ ]:
registry = PluginSystem.get_registered("incompressibleFluidModel")
assert registry is not None, "incompressibleFluidModel not imported"

print("Registered incompressibleFluid plugins:")
for plugin_cls in registry.plugin_registry:
    print(f"  - {plugin_cls.__name__}")

## Prepare a case for solver-level introspection
``create_init`` produces a ``StagedInit`` bound to a case directory.
Both ``solver_inputs()`` and ``scheme_inputs()`` work on it without
running the LOAD/BUILD stages — they only need the case to discover
which optional models pass ``detect``.



In [ ]:
case = clone_case("pitzDaily")
init = create_init(case_dir=case)
print(f"case: {case}")

## Dump every config class the solver might consume
``solver_inputs()`` returns ``{name: ConfigClass}`` for two kinds of
input the solver needs to rebuild a case:

- **Per-model configs** — one per core model (Pimple/Simple/Piso)
  and per registered plugin (boussinesq, spalart_allmaras, ...).
- **Framework case-file configs** — declared on the solver itself
  via ``incompressibleFluid.register_input(...)``: controlDict,
  transportProperties, fvSchemes, fvSolution. These come from the
  solver's own contract with the case directory, not from any
  individual model.

Calling ``model_json_schema`` on any returned class gives a JSON
schema you can feed to a UI, a validator, or an LLM.



In [ ]:
configs = init.solver_inputs()

print(f"\n{len(configs)} configurable inputs:")
for name, cls in configs.items():
    fields = ", ".join(cls.model_fields.keys()) or "(no fields)"
    bound_file = getattr(getattr(cls, "io_config", None), "file", "—")
    print(f"  {name:24s}  {cls.__name__:28s}  file: {bound_file}")
    print(f"  {'':24s}  fields: {fields}")

## Inspect one config in detail
Pick any entry from ``configs`` and print the full schema. The
``properties`` block has one entry per field with its type,
default, and any ``Field(...)`` constraints (``gt``, ``le``, etc.).



In [ ]:
example_name = next(iter(configs))
schema = configs[example_name].model_json_schema()
print(f"\nJSON schema for '{example_name}':")
print(json.dumps(schema, indent=2))

## Dump the fvSchemes / fvSolution surface
``scheme_inputs()`` walks every active operation, collects its
``@fvSchemes.add`` / ``@fvSolution.add`` declarations, and builds a
typed Pydantic model whose fields *are* the required scheme entries.
Each field is typed with the proper scheme union (``DdtScheme``,
``DivScheme``, …) so the same model both documents the requirement
and validates user input.



In [ ]:
scheme_model = init.scheme_inputs()
print(f"\nScheme surface: {scheme_model.__name__}")
for field_name, info in scheme_model.model_fields.items():
    annotation = getattr(info.annotation, "__name__", str(info.annotation))
    print(f"  {field_name:24s}  {annotation}")

## Combined "what does this solver need from me?" report
A single dict that captures the typed surface of the solver for the
current case. Drop it into a file and you have a portable, machine-
checkable contract — useful for templating new cases, generating
input forms, or feeding to an AI assistant.



In [ ]:
report = {
    "models": {name: cls.model_json_schema() for name, cls in configs.items()},
    "schemes": scheme_model.model_json_schema(),
}

report_path = Path(case) / "solver_surface.json"
report_path.write_text(json.dumps(report, indent=2))
print(f"\nWrote full surface to {report_path}")
print(f"  ({report_path.stat().st_size // 1024} KiB)")

## When to use which
- ``PluginSystem.get_registered(...).plugin_registry`` — "what
  *could* be active?" Works with no case, no detection. Use it
  when building a model picker UI or listing what a package
  contributes.
- ``init.solver_inputs()`` — "what *is* active for this case?"
  Reflects the result of ``@detect`` on every plugin, so it
  answers per-case questions. Use it in CI, in editors, and as
  the starting point for case templates.
- ``init.scheme_inputs()`` — "which ``fvSchemes`` / ``fvSolution``
  entries must this case provide?" Pair with
  :doc:`/how-to/validate-without-running` to fail fast on missing
  entries before launching the solver.

